<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union

## SQLAlchemy imports
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

# aux packages
from IPython.display import Image

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 5 {file_name}

# Extract 
 - parse text file into panadas file
 - ignore first row with date ; seen in the head output
    - set second row to header

In [ ]:
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
print(f"\nDataframe shape: {barra_df_07.shape}")
print("head")
display(barra_df_07.head(3))
print("tail")
display(barra_df_07.tail(3))

In [ ]:
# print DataFrame information showing index dtype and columns, non-NA values and memory usage.
barra_df_07.info()

In [ ]:
# show column names
barra_df_07.columns

In [ ]:
# clean column names; first remove leading and trailing white spaces
barra_df_07.columns = barra_df_07.columns.str.strip()

# check if the column name has invalid characters
invalid_characters = {"%": "_pct"}

# check if column begins with a number; if so, prepend with "col_"
def check_name_leads_with_number(column_name):
    if column_name[0].isdigit():
        print(f"Column name '{column_name}' begins with a number. Prepending with 'col_'.")
        return "col_" + column_name
    else:
        return column_name

# clean column names; replace invalid characters and check if column name begins with a number
for invalid_char, replacement in invalid_characters.items():
    barra_df_07.columns = barra_df_07.columns.str.replace(invalid_char, replacement, regex=False)
barra_df_07.columns = barra_df_07.columns.map(check_name_leads_with_number)

# make lowercase
barra_df_07.columns = barra_df_07.columns.str.lower()


In [ ]:
# new column names
barra_df_07.columns

In [ ]:
# Descriptive statistics include those that summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values.
barra_df_07.describe()

In [ ]:
# box plot numeric columns; find them first

def find_numeric_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"found {len(numeric_cols)} numeric columns in the DataFrame.")
    print(f"Numeric columns: {numeric_cols}")
    
    return numeric_cols

numeric_columns = find_numeric_columns(barra_df_07)

# to fix the the stack of boxplots we define rows
num_rows = math.ceil(len(numeric_columns) / 4) # 4 columns per row


# melt the df; so that it is not on the same axis
df_melted = barra_df_07.melt(
                            value_vars=numeric_columns,
                            var_name="factor_type", 
                            value_name="factor_value"
                            )


fig = px.box(
    df_melted,
    y="factor_value", # numeric column to plot
    facet_col="factor_type", # categorical column to facet by
    facet_col_wrap=4, # like a line break 
    height = 300 * num_rows, # adjust height based on number of rows needed
    template='plotly_dark',
    points='outliers', # show outliers as points
    facet_col_spacing=0.02, # reduce spacing between facets
    facet_row_spacing=0.02, # reduce spacing between rows of facets
)

fig.update_yaxes(matches=None, showticklabels=True) # allow y axes to have different scales



#clean title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


#fig.show()

# Transform (Data Cleaning)
 - column names - trim leading and trailing white spaces -- done earlier
    - (optional): set make column names lowercase
 - use a dictionary to replace invalid characters and the value with its replacement (i.e. {% : "_pct"})
 - identify how NaN values are encoded (from box plot hbta has -999)
 - set index

In [ ]:
barra_df_07.head(3)

In [ ]:
# display current indices
barra_df_07.index

In [ ]:
barra_df_07.set_index('ticker').index # by itself does not persist, need to assign back to df or use inplace=True
barra_df_07 = barra_df_07.set_index('ticker') # assign back to df to persist the change

In [ ]:
# find non numbers by column
# numeric_columns already holds numeric columns
non_numeric_columns = [col for col in barra_df_07.columns if col not in numeric_columns]
non_numeric_columns

#optionally: barra_df_07.select_dtypes('object')

In [ ]:
# barra_df_07.loc['AAPL'] error
# barra_df_07[non_numeric_columns].str.strip() # does not work because str returns a series. need to apply to columns separately

def strip_value(value: pd.Series) -> str:
    return value.str.strip()

barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(strip_value)

# lambda way
# barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(lambda x: x.str.strip())

In [ ]:
barra_df_07.head(3)

In [ ]:
print(barra_df_07.index[:5])
print(barra_df_07.index.name)

In [ ]:
# apply strip function to index
barra_df_07.index = barra_df_07.index.str.strip()
barra_df_07.index[:5]

### Troubleshooting key error
```python
# look at first/last labels of the index
print(df.index[:5])
# check if ticket is the index
print(df.index.name)
```

```text
Index(['IX    ', 'SAOL  ', 'IXYS  ', 'CDGT  ', 'CVRG  '], dtype='object', name='ticker')
ticker
```

white spaces in ticker

confirm with substring search
```python
aapl_filter = df[df.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]
```

```text
#STDOUT
Index(['AAPL  '], dtype='object', name='ticker')
```

In [ ]:
aapl_filter = barra_df_07[barra_df_07.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]

In [ ]:
barra_df_07.loc['AAPL']

## look for null values
- -999 is being used to encode null values
    - search in the numeric columns, return the column

In [ ]:
# intially was going to search all columns, but -999 is numeric so we can just search numeric columns for -999
cols_with_nulls = barra_df_07.columns[(barra_df_07 == -999).any()]
cols_with_nulls

In [ ]:
# null counts
null_counts = (barra_df_07['hbta'] == -999).sum()
null_counts

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)

In [ ]:
# earlier we saw the hbta box plot with the -999 values
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN'
)
fig.show()

### SQL Books/References 
- SQL Queries for Mere Mortals (John Viescas) 
- Using SQLite (Jay Kreibich / O'Reilly) 
- SQLAlchemy 2 in practice (Miguel Grinberg)

# LOAD
- load into SQL

## Understanding SQL
A database exists to collect and store data in some organized manner for a specific purpose.
Generally, there are two types of databases:
- operational : used to collect, modify and maintain data on a day to day basis.
- analytical : stores and tracks historical and time dependent data. 
    - Data tends to be static -- meaning it is never/rarely modified. New data is often added.

The Father of the relational model, is Dr. Edgar F. Codd. A relation is a table in set theory and the Relational model is based on 
two branches of mathematics:
- 1. Set theory
- 2. First-order predicate logic -- Filters

The father of data warehousing was William (Bill) H. Immon who developed the idea that organizations would access data stored in any number of non-relational databases.

## Anatomy of a Relational Database
Data in a relational database is stored in, you guesssed it, relations. Relations are tables composed of tuples and attributes. 
- Tuples are another way of saying records or rows. 
- Attributes are the fields and columns in a table.
    - Columns is a structure that stores data. It represents a characteristic of the subject of the table.

A table should always represent a single specific subject.
Tables can represent:
- an object: tangible; (think noun) -- person, place or thing.
- event: occurs at a given point in time aand has characteristics you wish to record.

We retrieve data from columns, and if they are properly designed, it only contains only one value.
Rows represent a unique instance of the subject of a table.
- a row is composed of the entire set of columns in a table.

Keys: one or more columns that uniquely identify each row within a table.
- A primary key is important for two reasons, in addition to enforcing table-level integrity and helps establish relationships with
other data.
    - 1. its value represents a specific row throughout the entire database
    - 2. it's column identifies a given table throughout the entire database.

Foreign Keys: take a copy of the primary key form the first table and insert into a second table -- becoming a foreign key. They help to ensure relationship level integrity. Rows in both tables will always be properly related because the value of a foreign key must be drawn from the values of the primary key to which it refers to.
- the second table already has a primary key of its own, and the primary key you are introducing from the first table is foreign to the second table.

Views: are virtual tables composed of columns from one or more tables in the databse. 
- views can be thought of a saved query because only the structure is stored.

### Relationships
- one-to-one: a table has been split into two parts.
- one-to-many: the 'one' has a relationship with many rows in the "many" side.
- many-to-many: these require a 'linking table'. Linking tables are a way of associating rows from one table with those of th oether. The advantage is that it allows you to associate any number of rows from both tables in the relationship.

There is a difference between:
- Database theory: rules and theory that form the basis of the relational model
- Database design: structured, organized set of processes used to design a relational database. It works to ensure integrity, consistency, and accuracy of the data.

# SQLALCHEMY

```python
pip install sqlalchemy
```
SQLAlchemy library is divided into two modules: CORE and ORM (Object Relational Mapping).

Core: contains the database integration logic for supported database dialects, collection of classes to describe database tables, and a system for generating SQL statements using Python language constructs.

ORM: an abstraction layer between Python and the database that allows database operations to be derived from actions performed on Python objects.

## The Database Engine:
SQLAlchemy uses 'engine' objects to manage connections to a database. The create_engine() function constructs an engine given a database URL.

## SQLITE
SQLite is a C-language library that implements a self-contained relational database.

In [ ]:
def get_schema(df: pd.DataFrame) -> dict:
    """
    loop through a dataframe looking at each column and get its dtype name and length
    """
    schema_dictonary = dict()
    for col in df.columns:
        dtype_name = df[col].dtype.name

        # create a nest dictionary for each column with dtype and length
        column_info = {"dtype": dtype_name}
        
        if dtype_name == 'object':
            # get the max length of strings in the column
            max_len = df[col].astype(str).str.len().max()
            column_info["max_length"] = int(max_len) if not pd.isna(max_len) else 0

        elif dtype_name in ['int64', 'float64']:
            unique_values = set(df[col].dropna().unique())
            if unique_values.issubset({0, 1}):
                column_info["dtype"] = 'boolean'

        schema_dictonary[col] = column_info
    return schema_dictonary
            

In [ ]:
type_dict = get_schema(barra_df_07)
type_dict

In [ ]:
# loop through type dictionary and show string columns with their max length
for col, info in type_dict.items():
    if info['dtype'] == 'object':
        print(f'{col} has max length of {info["max_length"]}')

In [ ]:
# String Definitions, adding 50% buffer and rounding up with math.ceil
SQL_TYPE_MAPPING = {
    'int64': Integer,
    'float64': Float,
}

def generate_sqlalchemy_schema(meta_data : dict) -> dict:
    """
    given a dictionary with col as keys and a nested dict with dtype and max length,
    we build a new dictionary that adds a 50% buffer to string lengths,
    if it is int64 or float64 we use a SQL_TYPE_MAPPING constants to use sqlalchemy types,
    we previously labeled columsn with values between 0 and 1 as boolean, so this handles it.   
    
    """
    
    orm_schema = dict()

    for col, info in meta_data.items():
        dtype = info['dtype']
        
        match dtype:
            case 'object':
                max_length = info['max_length']
                buffer_length = math.ceil(max_length * 1.5) # add 20% buffer and round up
                orm_schema[col] = String(buffer_length)
            case 'int64' | 'float64':
                orm_schema[col] = SQL_TYPE_MAPPING[dtype]
            case 'boolean':
                orm_schema[col] = Boolean
            case _:
                print(f"Data type {dtype} for column {col} not recognized.")
                
    return orm_schema


In [ ]:
dtype_dict = generate_sqlalchemy_schema(type_dict)
dtype_dict

### NOTE ON CONNECTION ENGINE
SQLITE:
- 3 slashes is relative paths
- 4 slashes is for absolute paths <br>
reference: https://docs.sqlalchemy.org/en/21/core/engines.html#sqlite

In [ ]:
# load dataframe to SQL and keep index because we set it in the df
# wrap it around an if statement to check for file existence;
# if file exists skip

db_name = 'barra_database.db'
engine = create_engine(f'sqlite:///{db_name}')
db_path = curr_working_dir / db_name
if not db_path.exists():
    print(f"Database {db_path} does not exist. Loading DataFrame to SQL.")
    barra_df_07.to_sql(
        name='barra_factors',
        con=engine,
        if_exists='fail',
        index=True,
        dtype=dtype_dict
    )
else:
    print(f"Database {db_path} already exists. Skipping loading DataFrame to SQL.")

# SQL Queries for Barra Factor Analysis

In [ ]:
image_folder = curr_working_dir / 'src' / 'images'
Image(image_folder / 'SELECT_STATEMENT_mere-mortals.jpg', width=600)

# SQL Queries for Barra Factor Analysis

## 1. Basic Count Query
```sql
SELECT COUNT(*) as total_records
FROM barra_factors;
```
This query gives us the total number of records in our database. It's useful for verifying that all data was loaded correctly.

TRANSLATION: COUNT all rows, name it total_records, FROM barra_factors


In [ ]:
count_query = """
SELECT
    COUNT(*) as total_records
FROM barra_factors;
"""

In [ ]:
db_records = pd.read_sql(count_query, engine)
print(f'Total records:\n {db_records}')

In [ ]:
# inspect the struc of db_records
db_records
print(f'type of db_records: {type(db_records)}')

In [ ]:
# assert that database records match dataframe records
assert len(barra_df_07) == db_records['total_records'][0]
print(f'barra_df_07 records: {len(barra_df_07)}, db_records: {db_records["total_records"][0]}. Assertion passed, record counts match.')


## 2. Summary Statistics Query
```sql
SELECT
    COUNT(*) as count,
    AVG(BETA) as avg_beta,
    AVG(SIZE) as avg_size,
    AVG(MOMENTUM) as avg_momentum,
    AVG(VALUE) as avg_value
FROM barra_factors;
```
This query calculates average values for key Barra factors. Common aggregate functions include:
- `AVG()`: Average value
- `STDDEV()`: Standard deviation
- `MIN()`: Minimum value
- `MAX()`: Maximum value


In [ ]:
Image(image_folder / 'AGG_FUNCS_mere-mortals.jpg', width=600)

In [ ]:
stats_query = '''
SELECT
    COUNT(*) as count,
    AVG(beta) as avg_beta,
    AVG(size) as avg_size,
    AVG(momentum) as avg_momentum,
    AVG(value) as avg_value
FROM barra_factors;    
'''
stats_results = pd.read_sql(stats_query, engine)
print(f'Summary statistics from SQL query:\n {stats_results}')

In [ ]:
Image(image_folder / 'GROUP_BY-mere-mortals.jpg', width=600)

# GROUP BY 

```sql
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
```
This query demonstrates:
- `GROUP BY`: Aggregating data by category
- `ORDER BY`: Sorting results
- `LIMIT`: Restricting number of results


GROUP BY
* **Collapses** rows into summary statistics
* Returns one row per group
* Loses individual row details


## Grouping Data
GROUP BY forms subsets of rows based on matching values. It allows you to choose the columns from a table result letting the database use them as definitions for how to group the rows.
- Rows that have the same value in the list of chosen columns are collected into a group. 
- GROUP BY cannot be applied to an expression (a logical statement that evaluates to true/false/unknown)

In [ ]:
group_by_indname_query_by_count = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS count,
    AVG(capitalization) AS avg_capitalization
FROM barra_factors
GROUP BY indname1
ORDER BY count DESC
LIMIT 5;
'''
group_by_count_results = pd.read_sql(group_by_indname_query_by_count, engine)
print(f'Top 5 Industry Distributions by count:\n {group_by_count_results}')

In [ ]:
group_by_indname_by_cap_query = '''
SELECT
    indname1 AS industry,
    COUNT(*) AS count,
    AVG(capitalization) AS avg_capitalization
FROM barra_factors
GROUP BY indname1
ORDER BY avg_capitalization DESC
LIMIT 5;
'''
group_by_cap_results = pd.read_sql(group_by_indname_by_cap_query, engine)
print(f'Top 5 Industry Distributions by average capitalization:\n {group_by_cap_results}')

## Window Functions

* **Maintains** all rows
* Adds calculations to each row
* Can see individual values alongside aggregates

Window functions allows for working with data and its adjacent rows allowing for operations such as generating running totals.
A window function:
- maps a vector to a function and produces a vector of the same length.
    - in contrast, an aggregate function reduces a vector to a scalar.
- aggregated data is replaced by sub-total rows preserving the details of the data that went into the aggregation functions

OVER can be thought as a context manager that defines the scope (the data) the function can see. OVER applies a function and broadcast it across a given vector of rows.
Context managers exist to provide a safe container for an operation to happen without affecting the rest of the data -- defines the visibilty.
SQL context managers manage the 'data window' solving the problem of an aggregate function being global, meaning it can see everything and collapses into one number. OVER sets up a temporary scope allowing for everything inside this clause to happen under a specific set of rules, once completed, the rules disappear.

WHERE filters down to one group.

In [ ]:
schema_info_query = pd.read_sql("PRAGMA table_info('barra_factors');", engine)

size_query_limit_5 = '''
SELECT
    ticker,
    size
FROM barra_factors
ORDER BY size DESC
LIMIT 5;
'''
print(f'5 stocks with size factor values:\n {pd.read_sql(size_query_limit_5, engine)}')




In [ ]:
# print liquid stocks
liquid_stocks_query_estuni_gt_1 = '''
SELECT
    name
FROM barra_factors
WHERE e3estu = TRUE AND size > 1 --- Liquid US stocks within ESTUNI and size greater than 1
ORDER BY size DESC;
'''
liquid_stocks_query_estuni_gt_1_results = pd.read_sql(liquid_stocks_query_estuni_gt_1, engine)
print(f'Liquid stocks WHERE ESTUNI = True and size > 1:\n {liquid_stocks_query_estuni_gt_1_results}')

In [ ]:
# industry distibution of liquid stocks
industry_dist_liquid_stocks_query = '''
SELECT
    name,
    indname1 AS industry,
    leverage,
    AVG(leverage) OVER (PARTITION by INDNAME1) AS avg_industry_leverage,
    leverage - AVG(leverage) OVER (PARTITION by INDNAME1) as diff_from_avg
FROM barra_factors
WHERE e3estu = TRUE AND size > 1 --- Liquid US stocks within ESTUNI and size greater than 1
ORDER BY size DESC;
'''
industry_dist_liquid_stocks_query_results = pd.read_sql(industry_dist_liquid_stocks_query, engine)
print(f'Industry distribution of liquid stocks WHERE ESTUNI = True and size > 1:\n {industry_dist_liquid_stocks_query_results}')

In [ ]:
# liquid us banks within ESTUNI and size greater than -1
liquid_us_banks_query_estuni_gt_neg1 = '''
SELECT
    name,
    leverage,
    AVG(leverage) OVER (PARTITION by indname1) AS avg_industry_leverage,
    leverage - AVG(leverage) OVER (PARTITION by indname1) as diff_from_avg
FROM barra_factors
WHERE e3estu = TRUE AND size > -1 AND indname1 = 'BANKS' --- Liquid US banks within ESTUNI and size greater than -1
ORDER BY size DESC;
'''
liquid_us_banks_query_estuni_gt_neg1_results = pd.read_sql(liquid_us_banks_query_estuni_gt_neg1, engine)
print(f'Liquid US banks within ESTUNI and size greater than -1:\n {liquid_us_banks_query_estuni_gt_neg1_results}')

# DISTINCT
To get unique industry names (INDNAME1), you can use the DISTINCT keyword in SQL. Here's how:

In [ ]:
# query to get unique industry names
unique_industries_query = '''
SELECT
    DISTINCT indname1 AS industry
FROM barra_factors
ORDER BY industry;
'''
unique_industries_results = pd.read_sql(unique_industries_query, engine)
print(f'Unique industry names:\n {unique_industries_results}')

In [ ]:
# query to show how many unique industries exist
unique_industries_count_query = '''
SELECT
    COUNT(DISTINCT indname1) AS unique_industry_count
FROM barra_factors;
'''
unique_industries_count_results = pd.read_sql(unique_industries_count_query, engine)
print(f'Number of unique industries:\n {unique_industries_count_results}')

# Understanding SQL Filtering with WHERE and AND

Let's explore how to filter data in SQL using a practical example with our Barra dataset. We'll look at how to find specific industries and add additional conditions.

## Basic Industry Query
First, let's look at our base query:
```python
# Basic industry distribution
basic_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(basic_industry_query, engine)
```

## Adding WHERE Clause
Now let's add filtering to focus on large-cap stocks:
```python
# Adding WHERE to filter results
filtered_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
WHERE CAPITALIZATION > 1000000000  -- Filter for large cap stocks > $1bn
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(filtered_industry_query, engine)
```

## Using Multiple Conditions with AND
Let's add more conditions to find large-cap stocks with positive momentum:
```python
# Using WHERE with multiple conditions
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""

pd.read_sql(complex_industry_query, engine)
```

## Understanding SQL Filtering
1. **WHERE Clause**:
   - Acts like a filter on your data
   - Goes after the FROM statement
   - Only rows that meet the condition are included
   - Example: `WHERE CAPITALIZATION > 1000000` only shows large-cap stocks

2. **AND Operator**:
   - Combines multiple conditions
   - ALL conditions must be true for the row to be included
   - Think of it like a series of filters
   - Example: `WHERE CAPITALIZATION > 1000000 AND MOMENTUM > 0` shows only large-cap stocks with positive momentum

3. **Order of Operations**:
   ```sql
   SELECT columns           -- 3. Select specific columns
   FROM table              -- 1. Start with table
   WHERE conditions        -- 2. Filter rows
   GROUP BY column         -- 4. Group results
   ORDER BY column         -- 5. Sort results
   LIMIT number;           -- 6. Limit number of rows
   ```


In [ ]:
Image(image_folder / 'WHERE-mere-mortals.jpg', width=600)

In [ ]:
Image(image_folder / 'AND_OPERATOR-mere-mortals.jpg', width=600)

In [ ]:
# Basic industry query
basic_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
basic_industry_query_results = pd.read_sql(basic_industry_query, engine)
print(f'Industry distribution:\n {basic_industry_query_results}')



In [ ]:
# adding WHERE clause
filtered_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap
FROM barra_factors
WHERE CAPITALIZATION > 1000000000  -- Filter for large cap stocks > $1bn
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
filtered_industry_query_results = pd.read_sql(filtered_industry_query, engine)
print(f'Industry distribution for large cap stocks:\n {filtered_industry_query_results}')

In [ ]:
# using multiple conditions with AND
complex_industry_query = """
SELECT
    INDNAME1,
    COUNT(*) as count,
    AVG(CAPITALIZATION) as avg_cap,
    AVG(MOMENTUM) as avg_momentum
FROM barra_factors
WHERE CAPITALIZATION > 1000000000     -- First condition
    AND MOMENTUM > 0               -- Second condition
    AND NONESTU = 0               -- Third condition
GROUP BY INDNAME1
ORDER BY count DESC
LIMIT 5;
"""
complex_industry_query_results = pd.read_sql(complex_industry_query, engine)
print(f'Industry distribution for large cap stocks:\n {complex_industry_query_results}')